# Notebook 08 — Concolic Exploration on CNNs

Runs Algorithm 1 (concolic exploration) on the CNN architectures using
the universal `concolic_engine_cnn.py` engine with autograd Jacobians.

| Model | Dataset | Est. Time |
|---|---|---|
| LeNet5 | MNIST | ~30 min |
| SmallCNN | CIFAR-10 | ~60 min |

Also runs the **activation pattern caching analysis** — the novel contribution
showing that pattern reuse across a test set amortises exploration cost.

**Requires:** `models/lenet5_mnist.pt`, `models/smallcnn_cifar.pt`

**Outputs:**
```
results/concolic_lenet5_mnist.json
results/concolic_smallcnn_cifar.json
results/cnn_cache_analysis.json
results/concolic_cnn_summary.json
```

In [1]:
import subprocess, sys
for pkg in ['torch', 'torchvision', 'numpy', 'scipy', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=False)
import torch
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.12.0+cpu


In [2]:
import sys, os, time
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

REPO_ROOT   = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

from utils.cnn_definitions      import load_cnn
from utils.concolic_engine_cnn  import (
    ConcolicExplorerCNN, run_concolic_batch_cnn,
    ActivationPatternCache
)
from utils.metrics import (
    get_correctly_classified_samples, save_results
)

MODELS_DIR  = REPO_ROOT / "models"
RESULTS_DIR = REPO_ROOT / "results"
DATA_DIR    = REPO_ROOT / "data"
RESULTS_DIR.mkdir(exist_ok=True)

SEED      = 42
N_SAMPLES = 100
NORM      = "linf"
MAX_ITER  = 100  # quality over quantity with correct directions

# Per-dataset max radius based on standard robustness literature:
# MNIST:   eps=0.3 in [0,1] space -> 0.3/0.3081 ~ 0.97 normalised
#          but we use 0.3 in normalised space (still generous)
# CIFAR:   eps=8/255 in [0,1] -> ~0.15 normalised
#          standard benchmark epsilon for CIFAR robustness
MAX_RADIUS_MNIST  = 0.3   # normalised space
MAX_RADIUS_CIFAR  = 0.15  # normalised space (~ 8/255 raw)

torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Config: norm={NORM}  max_iter={MAX_ITER}")
print(f"  LeNet5/MNIST radius: {MAX_RADIUS_MNIST}")
print(f"  SmallCNN/CIFAR radius: {MAX_RADIUS_CIFAR} (~ 8/255 standard)")


Config: norm=linf  max_iter=100
  LeNet5/MNIST radius: 0.3
  SmallCNN/CIFAR radius: 0.15 (~ 8/255 standard)


## 1 — Load datasets

In [3]:
mnist_tf = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
cifar_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])

def to_numpy(ds):
    l = DataLoader(ds, batch_size=len(ds), shuffle=False)
    X, y = next(iter(l))
    return X.numpy(), y.numpy()  # spatial (N,C,H,W)

print('Loading datasets...')
X_mnist, y_mnist = to_numpy(datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_tf))
X_cifar, y_cifar = to_numpy(datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_tf))
print(f'MNIST: {X_mnist.shape}  CIFAR: {X_cifar.shape}')

Loading datasets...
MNIST: (10000, 1, 32, 32)  CIFAR: (10000, 3, 32, 32)


## 2 — Run helper

In [4]:
def run_and_save(model_name, model, X_data, y_data, save_path,
                 max_radius=0.3):
    print(f'\n{"="*60}\n  {model_name}  (max_radius={max_radius})\n{"="*60}')

    X_sel, y_sel = get_correctly_classified_samples(
        model, X_data, y_data, N_SAMPLES, seed=SEED
    )
    print(f'  Samples selected: {len(X_sel)}')

    results, cache = run_concolic_batch_cnn(
        model, X_sel, y_sel,
        norm=NORM, max_iter=MAX_ITER,
        max_radius=max_radius, use_cache=True,
    )

    per_sample = [{
        "eps_upper"    : float(r.eps_upper),
        "eps_lower"    : float(r.eps_lower),
        "gap"          : float(r.eps_upper - r.eps_lower),
        "adv_found"    : r.adversarial_x is not None,
        "fgsm_fallback": r.fgsm_fallback,
        "cache_hit"    : r.cache_hit,
        "runtime_sec" : float(r.runtime_sec),
        "n_iterations": r.n_iterations,
    } for r in results]

    uppers  = np.array([r["eps_upper"]     for r in per_sample])
    lowers  = np.array([r["eps_lower"]     for r in per_sample])
    gaps    = uppers - lowers
    times   = np.array([r["runtime_sec"]   for r in per_sample])
    found   = np.array([r["adv_found"]     for r in per_sample])
    fgsm_fb = np.array([r["fgsm_fallback"] for r in per_sample])

    stats = {
        "model_name"                : model_name,
        "max_radius"                : max_radius,
        "mean_eps_upper"            : float(np.mean(uppers)),
        "std_eps_upper"             : float(np.std(uppers)),
        "mean_eps_lower"            : float(np.mean(lowers)),
        "std_eps_lower"             : float(np.std(lowers)),
        "mean_gap"                  : float(np.mean(gaps)),
        "fraction_adversarial_found": float(np.mean(found)),
        "fraction_fgsm_fallback"    : float(np.mean(fgsm_fb)),
        "mean_runtime_sec"          : float(np.mean(times)),
        "total_runtime_sec"         : float(np.sum(times)),
        "cache_hit_rate"            : float(cache.hit_rate),
        "n_distinct_patterns"       : cache.n_patterns,
        "n_samples"                 : len(per_sample),
        "max_iter"                  : MAX_ITER,
    }

    save_results({"per_sample": per_sample, "stats": stats,
                  "cache": cache.stats()}, str(save_path))

    print(f'  eps_upper : {stats["mean_eps_upper"]:.4f} +/- {stats["std_eps_upper"]:.4f}')
    print(f'  eps_lower : {stats["mean_eps_lower"]:.4f} +/- {stats["std_eps_lower"]:.4f}')
    print(f'  gap       : {stats["mean_gap"]:.4f}')
    print(f'  adv%      : {stats["fraction_adversarial_found"]*100:.1f}%  fgsm_fallback%={stats["fraction_fgsm_fallback"]*100:.1f}%')
    print(f'  time/s    : {stats["mean_runtime_sec"]:.2f}s')
    print(f'  cache     : hit_rate={stats["cache_hit_rate"]:.1%}  patterns={stats["n_distinct_patterns"]}')

    return {"per_sample": per_sample, "stats": stats, "cache": cache.stats()}

print("Run helper defined.")


Run helper defined.


## 3 — LeNet5 on MNIST
**Expected: ~30 min**

In [5]:
lenet  = load_cnn(str(MODELS_DIR / "lenet5_mnist.pt"))
r_lenet = run_and_save(
    "LeNet5-MNIST", lenet, X_mnist, y_mnist,
    RESULTS_DIR / "concolic_lenet5_mnist.json",
    max_radius=MAX_RADIUS_MNIST,
)


  Loaded ← D:\concolic_exploration\models\lenet5_mnist.pt  |  metadata: {'dataset': 'MNIST', 'input_size': 32, 'best_test_acc': 0.9918, 'architecture': 'LeNet5-ReLU-noBN', 'n_params': 82826}

  LeNet5-MNIST  (max_radius=0.3)
  Samples selected: 100
  [10/100] ε_upper=0.3000  ε_lower=0.3000  gap=0.0000  iters=1  cache=miss  time=48.418s | cache_hits=0/10
  [20/100] ε_upper=0.3000  ε_lower=0.3000  gap=0.0000  iters=1  cache=miss  time=41.444s | cache_hits=0/20
  [30/100] ε_upper=0.3000  ε_lower=0.1976  gap=0.1024  iters=100  cache=miss  time=41.097s | cache_hits=0/30
  [40/100] ε_upper=0.2111  ε_lower=0.1056  gap=0.1056  iters=100  cache=miss  time=34.836s | cache_hits=0/40
  [50/100] ε_upper=0.3000  ε_lower=0.3000  gap=0.0000  iters=1  cache=miss  time=39.423s | cache_hits=0/50
  [60/100] ε_upper=0.3000  ε_lower=0.2599  gap=0.0401  iters=100  cache=miss  time=38.863s | cache_hits=0/60
  [70/100] ε_upper=0.3000  ε_lower=0.2142  gap=0.0858  iters=100  cache=miss  time=29.771s | cache_hits

## 4 — SmallCNN on CIFAR-10
**Expected: ~60 min**

In [6]:
smallcnn = load_cnn(str(MODELS_DIR / "smallcnn_cifar.pt"))
r_cnn    = run_and_save(
    "SmallCNN-CIFAR10", smallcnn, X_cifar, y_cifar,
    RESULTS_DIR / "concolic_smallcnn_cifar.json",
    max_radius=MAX_RADIUS_CIFAR,
)


  Loaded ← D:\concolic_exploration\models\smallcnn_cifar.pt  |  metadata: {'dataset': 'CIFAR-10', 'input_size': 32, 'best_test_acc': 0.809, 'architecture': 'SmallCNN-3conv-ReLU-noBN', 'n_params': 620362}

  SmallCNN-CIFAR10  (max_radius=0.15)
  Samples selected: 100
  [10/100] ε_upper=0.1500  ε_lower=0.0299  gap=0.1201  iters=100  cache=miss  time=49.168s | cache_hits=0/10
  [20/100] ε_upper=0.1500  ε_lower=0.0361  gap=0.1139  iters=100  cache=miss  time=53.194s | cache_hits=0/20
  [30/100] ε_upper=0.1500  ε_lower=0.0116  gap=0.1384  iters=100  cache=miss  time=43.894s | cache_hits=0/30
  [40/100] ε_upper=0.1500  ε_lower=0.0094  gap=0.1406  iters=100  cache=miss  time=45.839s | cache_hits=0/40
  [50/100] ε_upper=0.1500  ε_lower=0.0157  gap=0.1343  iters=100  cache=miss  time=57.303s | cache_hits=0/50
  [60/100] ε_upper=0.1500  ε_lower=0.0515  gap=0.0985  iters=100  cache=miss  time=57.075s | cache_hits=0/60
  [70/100] ε_upper=0.1500  ε_lower=0.0661  gap=0.0839  iters=100  cache=miss  t

## 5 — Activation pattern caching analysis

In [8]:

MAX_RADIUS = MAX_RADIUS_MNIST

In [9]:
print('Caching analysis...')
print('(Compares runtime with vs without caching on LeNet5, 30 samples)\n')

X_sub, y_sub = get_correctly_classified_samples(
    lenet, X_mnist, y_mnist, 30, seed=SEED
)

# Without cache
t0 = time.time()
r_nocache, _ = run_concolic_batch_cnn(
    lenet, X_sub, y_sub,
    norm=NORM, max_iter=MAX_ITER, max_radius=MAX_RADIUS,
    use_cache=False,
)
time_nocache = time.time() - t0

# With cache
t0 = time.time()
r_cache, cache = run_concolic_batch_cnn(
    lenet, X_sub, y_sub,
    norm=NORM, max_iter=MAX_ITER, max_radius=MAX_RADIUS,
    use_cache=True,
)
time_cache = time.time() - t0

speedup = time_nocache / time_cache if time_cache > 0 else 1.0

cache_analysis = {
    'time_without_cache_sec' : time_nocache,
    'time_with_cache_sec'    : time_cache,
    'speedup'                : float(speedup),
    'cache_hit_rate'         : float(cache.hit_rate),
    'n_distinct_patterns'    : cache.n_patterns,
    'n_samples'              : len(X_sub),
    'pattern_reuse_rate'     : float(1.0 - cache.n_patterns / len(X_sub))
                               if len(X_sub) > 0 else 0.0,
}
save_results(cache_analysis, str(RESULTS_DIR / 'cnn_cache_analysis.json'))

print(f'  Without cache : {time_nocache:.1f}s')
print(f'  With cache    : {time_cache:.1f}s')
print(f'  Speedup       : {speedup:.2f}x')
print(f'  Hit rate      : {cache.hit_rate:.1%}')
print(f'  Distinct patterns: {cache.n_patterns} / {len(X_sub)} samples')
print(f'  Pattern reuse: {cache_analysis["pattern_reuse_rate"]:.1%} of samples reused a cached pattern')

Caching analysis...
(Compares runtime with vs without caching on LeNet5, 30 samples)

  [10/30] ε_upper=0.3000  ε_lower=0.3000  gap=0.0000  iters=1  cache=miss  time=37.592s | no cache
  [20/30] ε_upper=0.3000  ε_lower=0.3000  gap=0.0000  iters=1  cache=miss  time=29.131s | no cache
  [30/30] ε_upper=0.3000  ε_lower=0.1976  gap=0.1024  iters=100  cache=miss  time=29.112s | no cache
  [10/30] ε_upper=0.3000  ε_lower=0.3000  gap=0.0000  iters=1  cache=miss  time=29.337s | cache_hits=0/10
  [20/30] ε_upper=0.3000  ε_lower=0.3000  gap=0.0000  iters=1  cache=miss  time=33.128s | cache_hits=0/20
  [30/30] ε_upper=0.3000  ε_lower=0.1976  gap=0.1024  iters=100  cache=miss  time=29.117s | cache_hits=0/30
  Saved results → D:\concolic_exploration\results\cnn_cache_analysis.json
  Without cache : 977.0s
  With cache    : 910.4s
  Speedup       : 1.07x
  Hit rate      : 0.0%
  Distinct patterns: 30 / 30 samples
  Pattern reuse: 0.0% of samples reused a cached pattern


## 6 — Combined summary

In [10]:
summary = {
    'lenet5_mnist'  : r_lenet['stats'],
    'smallcnn_cifar': r_cnn['stats'],
    'cache_analysis': cache_analysis,
    'config': {'norm': NORM, 'max_iter': MAX_ITER,
               'max_radius': MAX_RADIUS, 'n_samples': N_SAMPLES},
}
save_results(summary, str(RESULTS_DIR / 'concolic_cnn_summary.json'))

print()
print('='*70)
print('  CNN CONCOLIC EXPLORATION SUMMARY')
print('='*70)
print(f'  {"Model":<22} {"eps_upper":>10} {"eps_lower":>10} {"gap":>8} {"adv%":>6} {"t/s":>6}')
print('-'*70)
for name, s in [('lenet5_mnist', r_lenet['stats']),
                ('smallcnn_cifar', r_cnn['stats'])]:
    print(f"  {name:<22}"
          f" {s['mean_eps_upper']:>10.4f}"
          f" {s['mean_eps_lower']:>10.4f}"
          f" {s['mean_gap']:>8.4f}"
          f" {s['fraction_adversarial_found']*100:>5.1f}%"
          f" {s['mean_runtime_sec']:>6.2f}s")
print('='*70)
print(f'\n  Cache speedup: {cache_analysis["speedup"]:.2f}x  '
      f'hit_rate={cache_analysis["cache_hit_rate"]:.1%}')
print('\n  Next -> run 09_full_analysis.ipynb')

  Saved results → D:\concolic_exploration\results\concolic_cnn_summary.json

  CNN CONCOLIC EXPLORATION SUMMARY
  Model                   eps_upper  eps_lower      gap   adv%    t/s
----------------------------------------------------------------------
  lenet5_mnist               0.2976     0.2594   0.0382   3.0%  34.79s
  smallcnn_cifar             0.1423     0.0317   0.1107   8.0%  67.16s

  Cache speedup: 1.07x  hit_rate=0.0%

  Next -> run 09_full_analysis.ipynb
